In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 指定模型ID
model_id = "Qwen/Qwen1.5-0.5B-Chat"

# 设置设备，优先使用GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 加载模型，并将其移动到指定设备
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

print("模型和分词器加载完成！")


d:\AI应用开发\langchain_learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
模型和分词器加载完成！


In [3]:
# 准备对话输入
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "你好，请介绍你自己。"}
]

# 使用分词器的模板格式化输入
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
print(f"分词后的文本是{text}")
# 编码输入文本
model_inputs = tokenizer([text], return_tensors="pt").to(device)

print("编码后的输入文本:")
print(model_inputs)


分词后的文本是<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
你好，请介绍你自己。<|im_end|>
<|im_start|>assistant

编码后的输入文本:
{'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198, 108386,  37945, 100157, 107828,
           1773, 151645,    198, 151644,  77091,    198]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:0')}


In [6]:
# 1. 获取输入 ID（取第 0 行，因为输入是 batch 格式）
ids = model_inputs['input_ids'][0]

# 2. 将 ID 转换成 Token 列表
tokens = tokenizer.convert_ids_to_tokens(ids)

# 3. 打印出来看对应关系
for i, (token_id, token_text) in enumerate(zip(ids, tokens)):
    # 这里的 token_text 可能会包含一些字节编码，需要解码成人类可读的中文
    decoded_text = tokenizer.decode([token_id])
    print(f"位置 {i:02d} | ID: {token_id:6d} | Token: {token_text:<10} | 对应中文: {decoded_text}")

位置 00 | ID: 151644 | Token: <|im_start|> | 对应中文: <|im_start|>
位置 01 | ID:   8948 | Token: system     | 对应中文: system
位置 02 | ID:    198 | Token: Ċ          | 对应中文: 

位置 03 | ID:   2610 | Token: You        | 对应中文: You
位置 04 | ID:    525 | Token: Ġare       | 对应中文:  are
位置 05 | ID:    264 | Token: Ġa         | 对应中文:  a
位置 06 | ID:  10950 | Token: Ġhelpful   | 对应中文:  helpful
位置 07 | ID:  17847 | Token: Ġassistant | 对应中文:  assistant
位置 08 | ID:     13 | Token: .          | 对应中文: .
位置 09 | ID: 151645 | Token: <|im_end|> | 对应中文: <|im_end|>
位置 10 | ID:    198 | Token: Ċ          | 对应中文: 

位置 11 | ID: 151644 | Token: <|im_start|> | 对应中文: <|im_start|>
位置 12 | ID:    872 | Token: user       | 对应中文: user
位置 13 | ID:    198 | Token: Ċ          | 对应中文: 

位置 14 | ID: 108386 | Token: ä½łå¥½     | 对应中文: 你好
位置 15 | ID:  37945 | Token: ï¼Įè¯·     | 对应中文: ，请
位置 16 | ID: 100157 | Token: ä»ĭç»į     | 对应中文: 介绍
位置 17 | ID: 107828 | Token: ä½łèĩªå·±  | 对应中文: 你自己
位置 18 | ID:   1773 | Token: ãĢĤ        | 对应中文: 。

In [19]:
text = "我爱北京天安门"
tokens =  tokenizer.tokenize(text)
# 分词期会先把中文的字按照UTF-8转为byte数值，然后再使用BPE将其转化为tiken,token在映射为ID
print(f"TO0KEN: {tokens}")
ids = tokenizer.encode(text)
ids

TO0KEN: ['æĪĳ', 'çĪ±', 'åĮĹäº¬', 'å¤©', 'å®ī', 'éĹ¨']


[35946, 99242, 68990, 35727, 50285, 64689]

In [20]:
decoded_id =  tokenizer.convert_tokens_to_ids(tokens)
decoded_id

[35946, 99242, 68990, 35727, 50285, 64689]

In [31]:
# 准备对话输入
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "你好！写一首诗"}
]

# 使用分词器的模板格式化输入
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
text2 =  "使用英文"

# 编码输入文本
model_inputs = tokenizer([text,text2], return_tensors="pt").to(device)

print("编码后的输入文本:")
print(model_inputs)

ValueError: Unable to create tensor, you should probably activate truncation and/or padding with 'padding=True' 'truncation=True' to have batched tensors with the same length. Perhaps your features (`input_ids` in this case) have excessive nesting (inputs type `list` where type `int` is expected).

In [29]:
# 使用模型生成回答
# max_new_tokens 控制了模型最多能生成多少个新的Token
generated_ids = model.generate(
    model_inputs.input_ids,
    max_new_tokens=512
)
print(f"{generated_ids.shape}")

# 将生成的 Token ID 截取掉输入部分
# 这样我们只解码模型新生成的部分
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

# 解码生成的 Token ID
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("\n模型的回答:")
print(response)


torch.Size([1, 118])

模型的回答:
你好，我来为您创作一首诗。

春风吹过桃花笑，绿草如茵映日长。
人间四月芳菲尽，此景唯有诗人知。

山川秀美水清流，一曲笛声悠扬中。
繁花似锦在眼前，犹如梦中的仙境。

世事无常心难定，世态炎凉情难言。
人生短暂若浮云，珍惜当下莫负年华。


In [35]:
from openai import OpenAI

client = OpenAI(
    api_key="ms-59f3b020-c728-45da-8d36-a7740722c2f2", # 请替换成您的ModelScope Access Token
    base_url="https://api-inference.modelscope.cn/v1/"
)


response = client.chat.completions.create(
    model="Qwen/Qwen2.5-Coder-32B-Instruct", # ModelScope Model-Id
    messages=[
        {
            'role': 'system',
            'content': 'You are a helpful assistant.'
        },
        {
            'role': 'user',
            'content': '用python写一下快排'
        }
    ],
    stream=True
)

for chunk in response:
    print(chunk.choices[0].delta.content, end='', flush=True)

当然可以！快速排序（QuickSort）是一种高效的排序算法，采用分治法策略。以下是一个用Python实现的快速排序示例：

```python
def quicksort(arr):
    if len(arr) <= 1:
        return arr
    else:
        pivot = arr[len(arr) // 2]  # 选择中间元素作为基准
        left = [x for x in arr if x < pivot]  # 小于基准的元素
        middle = [x for x in arr if x == pivot]  # 等于基准的元素
        right = [x for x in arr if x > pivot]  # 大于基准的元素
        return quicksort(left) + middle + quicksort(right)

# 测试
if __name__ == "__main__":
    test_array = [3, 6, 8, 10, 1, 2, 1]
    print("原始数组:", test_array)
    sorted_array = quicksort(test_array)
    print("排序后的数组:", sorted_array)
```

### 解释
1. **基准选择**：在这个实现中，我们选择数组的中间元素作为基准（pivot）。
2. **分区**：将数组分为三个部分：
   - `left`：所有小于基准的元素。
   - `middle`：所有等于基准的元素。
   - `right`：所有大于基准的元素。
3. **递归排序**：对`left`和`right`部分递归地进行快速排序。
4. **合并结果**：将排序后的`left`、`middle`和`right`部分合并成一个完整的排序数组。

### 注意事项
- 这个实现虽然简洁，但在处理大量重复元素时效率不高，因为它会创建多个包含相同元素的列表。
- 在实际应用中，可以使用原地分区的方法来优化空间复杂度。

如果你需要一个原地分区的版本，可以参考以下代码：

```python
def quicksort_inplace(arr, lo

In [ ]:
# ReAct 提示词模板
REACT_PROMPT_TEMPLATE = """
请注意，你是一个有能力调用外部工具的智能助手。

可用工具如下:
{tools}

请严格按照以下格式进行回应:

Thought: 你的思考过程，用于分析问题、拆解任务和规划下一步行动。
Action: 你决定采取的行动，必须是以下格式之一:
- `{{tool_name}}[{{tool_input}}]`:调用一个可用工具。
- `Finish[最终答案]`:当你认为已经获得最终答案时。
- 当你收集到足够的信息，能够回答用户的最终问题时，你必须在Action:字段后使用 finish(answer="...") 来输出最终答案。

现在，请开始解决以下问题:
Question: {question}
History: {history}
"""

REACT_PROMPT_TEMPLATE.format("10","10","10")


KeyError: 'tools'

In [ ]:
from dotenv import load_dotenv,find_dotenv
import os
load_dotenv(find_dotenv(),override= True)
api_key = os.environ.get("SERPAPI_API_KEY")
print(api_key)
b = f"dhah{a}"
b.format([{'change':10}])

fcd12ab7a29ec207a9c0ece3c6e8fea4bc350760267a0e1072a2b3dfa675d336


KeyError: "'change'"

In [61]:
tool_name = f"{{tool_name}}"

# 如果你写 f"{tool_name}"，输出是 Search
# 如果你想让输出结果带括号：
print(f"The template is: {{{tool_name}}}")
# 输出: The template is: {Search}

# 如果你纯粹想打印字面量 {{tool_name}}：
print(tool_name)
# 输出: {tool_name}

The template is: {{tool_name}}
{tool_name}


In [66]:
# ReAct 提示词模板
REACT_PROMPT_TEMPLATE = """
请注意，你是一个有能力调用外部工具的智能助手。

可用工具如下:
{tools}

请严格按照以下格式进行回应:

Thought: 你的思考过程，用于分析问题、拆解任务和规划下一步行动。
Action: 你决定采取的行动，必须是以下格式之一:
- `{{tool_name}}[{{tool_input}}]`:调用一个可用工具。
- `Finish[最终答案]`:当你认为已经获得最终答案时。
- 当你收集到足够的信息，能够回答用户的最终问题时，你必须在Action:字段后使用 finish(answer="...") 来输出最终答案。

现在，请开始解决以下问题:
Question: {question}
History: {history}
"""

REACT_PROMPT_TEMPLATE.format(tools="10",question ="10", history = "10")


'\n请注意，你是一个有能力调用外部工具的智能助手。\n\n可用工具如下:\n10\n\n请严格按照以下格式进行回应:\n\nThought: 你的思考过程，用于分析问题、拆解任务和规划下一步行动。\nAction: 你决定采取的行动，必须是以下格式之一:\n- `{tool_name}[{tool_input}]`:调用一个可用工具。\n- `Finish[最终答案]`:当你认为已经获得最终答案时。\n- 当你收集到足够的信息，能够回答用户的最终问题时，你必须在Action:字段后使用 finish(answer="...") 来输出最终答案。\n\n现在，请开始解决以下问题:\nQuestion: 10\nHistory: 10\n'

In [79]:
import re

action = "Finish[1.2]"
match_obj = re.match(r"Finish\[\d+\.\d\]", action)

if match_obj:
    result = match_obj.group(0)
    print(result)

Finish[1.2]
